# Silver to Gold
This notebook aggregates and joins data from the Silver layer to produce analytics-ready datasets in the Gold layer.

## Step 5: Aggregating Orders Data
Generate daily order summaries for reporting.
- Provides a daily summary of orders for dashboards and business insights.


In [0]:
CREATE OR REFRESH MATERIALIZED VIEW order_table_gold      -- PREVIOUS SYNTAX: CREATE OR REFRESH LIVE TABLE...
AS 
SELECT 
  date(order_timestamp) AS order_date, 
  count(*) AS total_daily_orders
FROM LIVE.order_table_silver                                 -- References the full orders_silver streaming table
GROUP BY date(order_timestamp)

##Step 6: Aggregating Customer Counts by State
Count the number of active customers per state.
- Enables geographic analysis of customer distribution.

In [0]:
CREATE MATERIALIZED VIEW customer_counts_state           -- PREVIOUS SYNTAX: CREATE OR REFRESH LIVE TABLE...
COMMENT "Total active customers per state"
AS 
SELECT 
  state, 
  count(*) as customer_count, 
  current_timestamp() updated_at
FROM LIVE.customers_silver
GROUP BY state

## Step 7: Joining Orders and Customers

Create a live view of subscribed customer order emails.
- Provides a real-time view of subscribed customers and their orders for email marketing and notifications.

In [0]:
CREATE LIVE VIEW subscribed_order_emails_v         
AS 
SELECT 
  a.customer_id, 
  a.order_id, 
  b.email 
FROM LIVE.order_table_silver a
INNER JOIN LIVE.customers_silver b
ON a.customer_id = b.customer_id
WHERE notifications = 'Y'